In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from tqdm import tqdm
import sys
import tempfile
import urllib.request

import librosa

from transformers import AutoTokenizer, AutoFeatureExtractor, AutoConfig

from data.audio.lhotse import LibriSpeechLhotse, PeoplesSpeechLhotse
from melt.processing_melt import MELTProcessor
from melt.modeling_melt import MELTForConditionalGeneration
from melt.configuration_melt import MELTConfig

In [3]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Or any tokenizer you want
AUDIO_ENCODER_NAME = "facebook/w2v-bert-2.0"  # Or any feature extractor

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
feature_extractor = AutoFeatureExtractor.from_pretrained(AUDIO_ENCODER_NAME)
processor = MELTProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

audio_config = AutoConfig.from_pretrained(AUDIO_ENCODER_NAME)
text_config = AutoConfig.from_pretrained(LLM_MODEL_NAME)
config = MELTConfig(
    audio_encoder_config=audio_config,
    text_decoder_config=text_config,
)
config.audio_bos_token_id = processor.tokenizer.convert_tokens_to_ids(["<|audio_bos|>"])[0]

In [4]:
model = MELTForConditionalGeneration(config).to("mps")

In [5]:
# Audio sample URL for testing
AUDIO_SAMPLE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"
AUDIO_SAMPLE_URL_2 = "recording.flac"

# Download and load audio sample
def load_audio_sample(url):
    """Load audio sample from URL using librosa."""
    with urllib.request.urlopen(url) as response:
        audio_bytes = response.read()
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_path = tmp_file.name
    
    audio, sr = librosa.load(tmp_path, sr=16000)
    return audio

def load_audio_local(path):
    """Load audio sample from local path using librosa."""
    audio, sr = librosa.load(path, sr=16000)
    return audio

audio_sample = load_audio_sample(AUDIO_SAMPLE_URL)
print(f"Audio sample shape: {audio_sample.shape}")
print(f"Audio duration: {len(audio_sample) / 16000:.2f} seconds")
audio_sample_2 = load_audio_local(AUDIO_SAMPLE_URL_2)
print(f"Audio sample 2 shape: {audio_sample_2.shape}")
print(f"Audio 2 duration: {len(audio_sample_2) / 16000:.2f} seconds")

Audio sample shape: (144000,)
Audio duration: 9.00 seconds
Audio sample 2 shape: (364324,)
Audio 2 duration: 22.77 seconds


In [6]:
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing well, thank you!"},
]

# Apply chat template
text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

result = processor(text=text_with_template, return_tensors="pt").to("mps")

Raw text after applying chat template:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you!<|im_end|>
<|im_start|>assistant




In [15]:
out = model(**result)

In [16]:
audio_token = processor.audio_token
text = f"Transcribe the following audio: {audio_token}"

print(f"Input text: {text}")
print()

result = processor(text=text, audio=audio_sample, return_tensors="pt").to("mps")

Input text: Transcribe the following audio: <|AUDIO|>



In [20]:
print(result["input_ids"].shape)
print(result["input_features"].shape)
print(processor.tokenizer.decode(result["input_ids"][0]))

torch.Size([1, 458])
torch.Size([1, 452, 160])
Transcribe the following audio: <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|

In [17]:
out = model(**result)

In [23]:
audio_token = processor.audio_token
messages = [
    {
        "role": "user",
        "content": f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all.",
    },
]

text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

audios = [audio_sample, audio_sample, audio_sample]
result = processor(text=text_with_template, audio=audios, return_tensors="pt").to("mps")

Raw text after applying chat template:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
<|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.<|im_end|>
<|im_start|>assistant




In [24]:
out = model(**result)

RuntimeError: MPS backend out of memory (MPS allocated: 20.10 GiB, other allocations: 9.94 MiB, max allowed: 20.13 GiB). Tried to allocate 45.78 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [7]:
audio_token = processor.audio_token
text = f"Compare {audio_token} with {audio_token}"
audios = [audio_sample, audio_sample]

print(f"Input text: {text}")
print(f"Number of audios: {len(audios)}")
print()

result = processor(text=text, audio=audios, return_tensors="pt").to("mps")
out = model(**result)

Input text: Compare <|AUDIO|> with <|AUDIO|>
Number of audios: 2



In [8]:
list(out.keys())

['logits', 'past_key_values']